# 02 - Feature Engineering

**Goal**: Turn the two dirty CSVs into a clean, minimal, contract-locked dataset ready for training.

**What this notebook does**:
1. Re-load both files with whitespace stripped
2. Drop duplicates (remember: PortScan is 25% duplicates!)
3. Replace `Infinity` with `0.0`, drop any `NaN` rows
4. Normalize **all** column names to `snake_case`, locks in the feature-name contract
5. Map string labels → integers
6. **Your decision**: pick ~15 features from the candidate list
7. Save artifacts: `../models/feature_names.json`, `../models/metadata.json` (partial), `../data/processed/cleaned.parquet`

**Artifacts produced here become the data contract for the runtime.** If `flow_builder.py` computes a feature under a different name or different units, the model will silently predict garbage. Take the naming seriously.

In [1]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

DATA_DIR      = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR    = Path('../models')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DDOS_CSV     = DATA_DIR / 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
PORTSCAN_CSV = DATA_DIR / 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'

## 1. Load + strip whitespace

In [2]:
def load_cicids_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    df['source_file'] = path.name
    return df

ddos_df     = load_cicids_csv(DDOS_CSV)
portscan_df = load_cicids_csv(PORTSCAN_CSV)
combined = pd.concat([ddos_df, portscan_df], ignore_index=True)
print(f'Before cleaning: {len(combined):,} rows')

Before cleaning: 512,212 rows


## 2. Cleaning pipeline

Order matters:
1. Replace Infinity with 0.0 (rate is undefined for zero-duration flows, treating them as zero is conservative)
2. Drop NaN rows (only ~19 rows, negligible)
3. Drop duplicates (critical - PortScan has 25%)

In [3]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out[numeric_cols].replace([np.inf, -np.inf], 0.0)
    before_nan = len(out)
    out = out.dropna()
    dropped_nan = before_nan - len(out)
    before_dup = len(out)
    out = out.drop_duplicates().reset_index(drop=True)
    dropped_dup = before_dup - len(out)
    print(f'  Dropped NaN rows:        {dropped_nan:,}')
    print(f'  Dropped duplicate rows:  {dropped_dup:,}')
    return out

cleaned = clean(combined)
print(f'\nAfter cleaning: {len(cleaned):,} rows (was {len(combined):,})')
print(f'\nClass distribution after dedup:')
print(cleaned['Label'].value_counts())

  Dropped NaN rows:        19
  Dropped duplicate rows:  74,986

After cleaning: 437,207 rows (was 512,212)

Class distribution after dedup:
Label
BENIGN      218372
DDoS        128016
PortScan     90819
Name: count, dtype: int64


## 3. Label mapping (string → int)

Scikit-learn needs integer labels. We lock the mapping now so training and runtime agree on what `0`, `1`, `2` mean.

In [4]:
LABEL_MAP = {
    'BENIGN':   0,
    'DDoS':     1,
    'PortScan': 2,
}
CLASS_NAMES = ['benign', 'ddos', 'portscan']  # index = integer label

# Rename the original string column FIRST to avoid a name collision:
# the snake_case pass (next cell) would turn 'Label' -> 'label', which would
# clash with the integer column we're about to create below.
cleaned = cleaned.rename(columns={'Label': 'label_str'})
cleaned['label'] = cleaned['label_str'].map(LABEL_MAP)
assert cleaned['label'].notna().all(), 'Unmapped label found - check LABEL_MAP against unique values'
cleaned['label'] = cleaned['label'].astype(int)

cleaned[['label_str', 'label']].drop_duplicates()

,label_str,label
0,BENIGN,0
18597,DDoS,1
224558,PortScan,2


## 4. Normalize column names → `snake_case`

This is the **hardest-to-debug** step if it goes wrong. The runtime's `flow_builder.py` will emit features with snake_case names like `flow_packets_per_s`. The training data must use identical names or scikit-learn will silently predict on misaligned inputs.

In [5]:
def to_snake(name: str) -> str:
    n = name.strip().lower()
    n = n.replace('/s', '_per_s')       # 'Flow Bytes/s' → 'flow bytes_per_s'
    n = re.sub(r'[^a-z0-9]+', '_', n)   # any non-alnum → underscore
    return n.strip('_')

rename_map = {c: to_snake(c) for c in cleaned.columns}
cleaned = cleaned.rename(columns=rename_map)

# Preserve the original name ↔ snake_case mapping for audit trail
(MODELS_DIR / 'column_rename_map.json').write_text(
    json.dumps(rename_map, indent=2)
)

# Preview a few
for original, snake in list(rename_map.items())[:8]:
    print(f"  {original!r:<40} → {snake!r}")

  'Destination Port'                       → 'destination_port'
  'Flow Duration'                          → 'flow_duration'
  'Total Fwd Packets'                      → 'total_fwd_packets'
  'Total Backward Packets'                 → 'total_backward_packets'
  'Total Length of Fwd Packets'            → 'total_length_of_fwd_packets'
  'Total Length of Bwd Packets'            → 'total_length_of_bwd_packets'
  'Fwd Packet Length Max'                  → 'fwd_packet_length_max'
  'Fwd Packet Length Min'                  → 'fwd_packet_length_min'


## 5. Full column catalog (reference)

Browse this list before picking your features.

In [6]:
numeric_cols = cleaned.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'label']
print(f'{len(numeric_cols)} numeric feature columns available:\n')
for i, c in enumerate(numeric_cols, 1):
    print(f'  {i:>2}. {c}')

78 numeric feature columns available:

   1. destination_port
   2. flow_duration
   3. total_fwd_packets
   4. total_backward_packets
   5. total_length_of_fwd_packets
   6. total_length_of_bwd_packets
   7. fwd_packet_length_max
   8. fwd_packet_length_min
   9. fwd_packet_length_mean
  10. fwd_packet_length_std
  11. bwd_packet_length_max
  12. bwd_packet_length_min
  13. bwd_packet_length_mean
  14. bwd_packet_length_std
  15. flow_bytes_per_s
  16. flow_packets_per_s
  17. flow_iat_mean
  18. flow_iat_std
  19. flow_iat_max
  20. flow_iat_min
  21. fwd_iat_total
  22. fwd_iat_mean
  23. fwd_iat_std
  24. fwd_iat_max
  25. fwd_iat_min
  26. bwd_iat_total
  27. bwd_iat_mean
  28. bwd_iat_std
  29. bwd_iat_max
  30. bwd_iat_min
  31. fwd_psh_flags
  32. bwd_psh_flags
  33. fwd_urg_flags
  34. bwd_urg_flags
  35. fwd_header_length
  36. bwd_header_length
  37. fwd_packets_per_s
  38. bwd_packets_per_s
  39. min_packet_length
  40. max_packet_length
  41. packet_length_mean
  42. packe

## 6. Feature Selection

CICIDS2017 gives us ~78 numeric features. We want **~15**. Picking too many gives the Random Forest room to overfit on noise and makes the runtime feature-extraction in `flow_builder.py` more painful. Picking too few loses signal.

### How to think about it

A good feature *discriminates attacks from benign*. For our two attacks:

| Attack | Packet signature |
|---|---|
| **SYN flood (DDoS)** | High packet rate, many SYNs never ACK'd, small packets, tight inter-arrival times |
| **Port Scan** | Very short flows (1–2 packets), hitting many dest ports, no backward packets on closed ports → RST response |

Features that would flag those behaviors (candidates below) fall into 5 buckets.

### Candidate list (18 - pick ~15)

**📊 Flow volume** (pick 2–3)
- `flow_duration` - scans are very short; floods can be either
- `total_fwd_packets` - DDoS sends many; scans send few
- `total_backward_packets` - benign is bidirectional; scans/floods often one-way
- `total_length_of_fwd_packets` - total bytes sent forward

**⚡ Rate** (pick 2)
- `flow_bytes_per_s` - DDoS floods have high byte rate
- `flow_packets_per_s` - scans + DDoS have distinctive packet rates

**📦 Packet size** (pick 2–3)
- `fwd_packet_length_mean` - attack packets tend to be small and uniform
- `fwd_packet_length_std` - attacks are homogeneous; benign varies
- `bwd_packet_length_mean` - 0 for scans hitting closed ports
- `average_packet_size` - overall size signal

**⏱ Inter-arrival timing** (pick 1–2)
- `flow_iat_mean` - DDoS packets arrive evenly and fast
- `flow_iat_std` - attacks are regular, benign is bursty

**🚩 TCP flags** (pick 3–5)
- `syn_flag_count` - **key signature** for SYN flood + SYN scan
- `ack_flag_count` - benign handshakes complete; attacks don't
- `fin_flag_count` - benign closes cleanly; attacks rarely do
- `rst_flag_count` - scans trigger RSTs on closed ports
- `psh_flag_count` - data-transfer indicator (benign)

### Constraints

1. Every name must **exactly** match one of the columns printed in cell 5 (snake_case).
2. The runtime (`flow_builder.py`) has to compute each feature from live packets. Some are cheaper than others — `syn_flag_count` is trivial; `flow_iat_std` requires keeping a running standard deviation. Keep that in mind.
3. Don't include `destination_port`. It seems useful but at inference time it's both the target of an attack AND the ordinary port of the victim service (HTTP = 80 for both an Apache server and a DDoS victim). It can leak dataset-specific info.

**Edit the cell below.** The cells after it depend on `FEATURES` being non-empty — you won't be able to skip this step.

In [7]:
# =====================================================================
#  DECISION — pick ~15 features from the candidates above
#  Keep the category comments so future-you remembers why each is here
# =====================================================================

FEATURES: list[str] = [
    # --- Flow volume ---
    'flow_duration',
    'total_fwd_packets',
    'total_backward_packets',

    # --- Rate ---
    'flow_bytes_per_s',
    'flow_packets_per_s',

    # --- Packet size ---
    'fwd_packet_length_mean',
    'bwd_packet_length_mean',
    'average_packet_size',

    # --- Inter-arrival timing ---
    'flow_iat_mean',
    'flow_iat_std',

    # --- TCP flags ---
    'syn_flag_count',
    'ack_flag_count',
    'fin_flag_count',
    'rst_flag_count',
    'psh_flag_count'
]

assert len(FEATURES) >= 8,  f'Pick at least 8 features (you picked {len(FEATURES)})'
assert len(FEATURES) <= 20, f'Keep it under 20 (you picked {len(FEATURES)})'
print(f'You picked {len(FEATURES)} features:')
for f in FEATURES:
    print(f'  - {f}')

You picked 15 features:
  - flow_duration
  - total_fwd_packets
  - total_backward_packets
  - flow_bytes_per_s
  - flow_packets_per_s
  - fwd_packet_length_mean
  - bwd_packet_length_mean
  - average_packet_size
  - flow_iat_mean
  - flow_iat_std
  - syn_flag_count
  - ack_flag_count
  - fin_flag_count
  - rst_flag_count
  - psh_flag_count


## 7. Validate + build the feature matrix

In [8]:
# Catch typos before they become silent bugs
missing = [f for f in FEATURES if f not in cleaned.columns]
assert not missing, f'These features are not in the data: {missing}. Check spelling against cell 5.'

X = cleaned[FEATURES].astype(np.float64)
y = cleaned['label'].astype(int)

print(f'X: {X.shape}  dtype={X.dtypes.unique()}')
print(f'y: {y.shape}  classes={sorted(y.unique().tolist())}')
print()
print('Per-class row counts:')
for class_idx, name in enumerate(CLASS_NAMES):
    n = int((y == class_idx).sum())
    print(f'  {class_idx} ({name}): {n:,}')

X: (437207, 15)  dtype=[dtype('float64')]
y: (437207,)  classes=[0, 1, 2]

Per-class row counts:
  0 (benign): 218,372
  1 (ddos): 128,016
  2 (portscan): 90,819


## 8. Save artifacts

Three files are written:
- `../models/feature_names.json` — **the contract**. Runtime reads this to know the exact order of features to pass to `model.predict()`.
- `../models/metadata.json` — label mapping, class names, units, creation date. Will be extended by notebook 03 with training metrics.
- `../data/processed/cleaned.parquet` — the cleaned dataframe (parquet = fast to reload in notebook 03).

In [9]:
def units_for(name: str) -> str:
    '''Map each feature to its unit. Runtime must produce values in the same unit.'''
    if name.endswith('_per_s'):    return 'per_second'
    if '_iat_' in name or name.endswith('_iat'): return 'microseconds'
    if name == 'flow_duration':    return 'microseconds'
    if 'packet_length' in name or 'packet_size' in name or 'length_of' in name: return 'bytes'
    if name.endswith('_packets'):  return 'count'
    if name.endswith('_flag_count'): return 'count'
    return 'count'

feature_units = {f: units_for(f) for f in FEATURES}

metadata = {
    'feature_names': FEATURES,
    'feature_units': feature_units,
    'label_map': LABEL_MAP,
    'class_names': CLASS_NAMES,
    'dataset': 'CICIDS2017 (Friday DDoS + PortScan)',
    'n_samples': int(len(cleaned)),
    'created_at': datetime.now(timezone.utc).isoformat(),
    # filled in by 03_model_training.ipynb
    'model': None,
    'training_metrics': None,
    'severity_thresholds': None,
}

(MODELS_DIR / 'feature_names.json').write_text(json.dumps(FEATURES, indent=2))
(MODELS_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2))

out_parquet = PROCESSED_DIR / 'cleaned.parquet'
cols_to_save = FEATURES + ['label', 'label_str', 'source_file']
cleaned[cols_to_save].to_parquet(out_parquet, index=False)

print(f'Saved: {MODELS_DIR / "feature_names.json"}')
print(f'Saved: {MODELS_DIR / "metadata.json"}')
print(f'Saved: {out_parquet}  ({out_parquet.stat().st_size / 1024 / 1024:.1f} MB)')

Saved: ..\models\feature_names.json
Saved: ..\models\metadata.json
Saved: ..\data\processed\cleaned.parquet  (14.4 MB)


## Summary

At this point:
- Duplicates are gone, NaN is gone, Infinity is replaced
- Column names are snake_case everywhere
- Labels are integer-mapped
- Your chosen features are locked into `feature_names.json`
- Cleaned parquet is ready for notebook 03

**Next**: `03_model_training.ipynb` — train/val/test split, fit a Random Forest, save `detector.pkl` + `scaler.pkl`.